# Phase 7: Lotto 6/42 K-means descriptive exploration

**No predictive claims.** The existing `STOP_NO_REPRODUCIBLE_PREDICTIVE_SIGNAL` verdict remains unchanged. Historical 6/42 combinations are unordered sets. This notebook runs only on the Phase 4 training partition (902 records through November 12, 2022); the validation and locked test draws are not opened.

Run the first code cell to initialize the notebook. It discovers the repository root from the current directory or its parents; in Colab, it clones the default branch only when no usable checkout is present. It installs the editable project and notebook dependencies into the active kernel, then validates imports before analysis. Set MAGIC_LOTTO_ROOT to select a local checkout when the kernel starts elsewhere. Re-running it reuses the checkout without resetting files. Local Jupyter uses the same cell and does not clone.


In [ ]:
# MAGIC_LOTTO_BOOTSTRAP_V1
from pathlib import Path
import os
import subprocess
import sys

REQUIRED_IMPORTS = ('magic_lotto', 'numpy', 'pandas', 'matplotlib', 'sklearn', 'IPython', 'magic_lotto.interfaces.phase7_cli')
REQUIRED_PATHS = ('scripts/phase7_kmeans_642.py', 'requirements-phase7.txt', 'data/processed/lotto_6_42_cleaned_normalized.csv')
ROOT_MARKERS = ('README.md', 'pyproject.toml', 'src/magic_lotto/__init__.py', 'docs/REPOSITORY_STRUCTURE.md')
COLAB_CLONE_ROOT = Path('/content/Magic_Lotto')
REPOSITORY_URL = 'https://github.com/Baelfyre/Magic_Lotto.git'

def _is_magic_lotto_root(path):
    return all((path / marker).is_file() for marker in ROOT_MARKERS)

def _find_magic_lotto_root(start):
    start = Path(start).resolve()
    return next((path for path in (start, *start.parents) if _is_magic_lotto_root(path)), None)

try:
    import google.colab
except ModuleNotFoundError as error:
    if error.name not in {"google", "google.colab"}:
        raise
    IN_COLAB = False
else:
    IN_COLAB = True

REQUESTED_REF = os.environ.get("MAGIC_LOTTO_REF") or None
ROOT_OVERRIDE = os.environ.get("MAGIC_LOTTO_ROOT")
REPO_ROOT = _find_magic_lotto_root(Path(ROOT_OVERRIDE)) if ROOT_OVERRIDE else _find_magic_lotto_root(Path.cwd())
if ROOT_OVERRIDE and REPO_ROOT is None:
    raise FileNotFoundError(f"MAGIC_LOTTO_ROOT is not a Magic_Lotto checkout: {ROOT_OVERRIDE}")
if REPO_ROOT is None and IN_COLAB:
    if COLAB_CLONE_ROOT.exists():
        if not _is_magic_lotto_root(COLAB_CLONE_ROOT):
            raise RuntimeError(
                f"Existing Colab path is not a usable Magic_Lotto checkout: {COLAB_CLONE_ROOT}. "
                "This setup will not overwrite it."
            )
        REPO_ROOT = COLAB_CLONE_ROOT
    else:
        clone_ref = REQUESTED_REF or "main"
        valid_ref = subprocess.run(
            ["git", "check-ref-format", "--branch", clone_ref],
            capture_output=True, text=True, check=False,
        )
        if valid_ref.returncode:
            raise ValueError(f"MAGIC_LOTTO_REF is not a valid branch name: {clone_ref!r}")
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", "--single-branch", "--branch", clone_ref,
             REPOSITORY_URL, str(COLAB_CLONE_ROOT)],
            capture_output=True, text=True, check=False,
        )
        if cloned.returncode:
            raise RuntimeError(f"Could not clone Magic_Lotto branch {clone_ref!r}: {cloned.stderr.strip()}")
        if not _is_magic_lotto_root(COLAB_CLONE_ROOT):
            raise RuntimeError(f"The cloned path is not a usable Magic_Lotto checkout: {COLAB_CLONE_ROOT}")
        REPO_ROOT = COLAB_CLONE_ROOT

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find a Magic_Lotto checkout in the current directory or its parents. "
        "Set MAGIC_LOTTO_ROOT to a local checkout, or run this notebook in Google Colab."
    )
ROOT = REPO_ROOT.resolve()
os.chdir(ROOT)
sys.dont_write_bytecode = True
installed = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[notebook]"],
    capture_output=True, text=True, check=False,
)
if installed.returncode:
    raise RuntimeError(f"Could not install Magic_Lotto dependencies in this notebook kernel: {installed.stderr.strip()}")
root = ROOT

from magic_lotto.interfaces.notebook_bootstrap import validate_notebook_environment
BRANCH, COMMIT = validate_notebook_environment(
    ROOT, REQUIRED_IMPORTS, REQUIRED_PATHS, IN_COLAB, REQUESTED_REF, subprocess.run
)
print(f"Magic_Lotto root: {ROOT}")
print(f"Checkout: {BRANCH or 'unavailable'} @ {COMMIT or 'unavailable'}")
print(f"Notebook imports validated: {', '.join(REQUIRED_IMPORTS) or 'standard library only'}")


In [ ]:
from IPython.display import display, Image, Markdown
import json
from magic_lotto.interfaces.phase7_cli import run

root = ROOT
out = root / 'reports' / 'phase7_kmeans_642'
result = run(root / 'data/processed/lotto_6_42_cleaned_normalized.csv', out, null_replicates=100, seed=162)
display(result)


## Graphs

The charts display empirical variation, pair co-occurrence differences, the relative composition of K-means groups, PCA projection, an elbow chart, and the fair-null inertia comparison. Seeing groups does **not** establish that particular numbers become more likely in later draws.


In [ ]:
titles = ['Number frequency', 'Pair residual heatmap', 'Cluster ball membership', 'PCA projection', 'Elbow plot', 'Fair-null inertia comparison']
names = ['01_frequency.png', '02_pair_residuals.png', '03_cluster_profiles.png', '04_cluster_pca.png', '05_elbow.png', '06_null_inertia.png']
for title, name in zip(titles, names):
    display(Markdown('### ' + title))
    display(Image(filename=str(out / name)))


## Evidence summary

The last cell prints the results **actually generated by the Python run**. Simulation p-values refer only to the descriptive lower-inertia diagnostic for fixed k=6. They are not a jackpot probability, an independent prediction test, or evidence of future advantage.


In [ ]:
print((out / 'README.md').read_text(encoding='utf-8'))
print(json.dumps(json.loads((out / 'summary.json').read_text(encoding='utf-8')), indent=2))
